# Project Chronos: P-CTC message test (Colab)
Self-contained. Runs on Cirq's simulator, which is what Colab provides: Colab has no quantum hardware. The past register `r` is measured **before** the message bit `m` is created by a quantum coin.

In [ ]:
!pip -q install cirq-core

In [ ]:
import cirq, numpy as np
A, B, R, Q, C = cirq.LineQubit.range(5)
past = cirq.Circuit(
    cirq.H(A), cirq.CNOT(A, B),          # entangled pair standing in for the time loop
    cirq.CNOT(A, R), cirq.measure(R, key='r'))   # PAST: record fixed here
future = cirq.Circuit(
    cirq.H(C), cirq.CNOT(C, Q),          # FUTURE: quantum coin writes message bit m
    cirq.CNOT(Q, B), cirq.H(Q),
    cirq.measure(Q, B, key='bell'), cirq.measure(C, key='m'))
circuit = past + future                  # every 'future' gate comes after the 'r' measurement
print(circuit)

In [ ]:
shots = 200_000
noisy = cirq.DensityMatrixSimulator(noise=cirq.ConstantQubitNoiseModel(cirq.depolarize(0.003)), seed=1)
res = noisy.run(circuit, repetitions=shots)
r = res.measurements['r'][:, 0]; m = res.measurements['m'][:, 0]
herald = (res.measurements['bell'].sum(axis=1) == 0)
acc = (r == m).mean(); se = 0.5 / np.sqrt(shots)
print(f"Unconditional accuracy (the real test): {acc:.4f}  (chance = 0.5, +/- {se:.4f})")
print(f"Herald rate: {herald.mean():.4f}")
print(f"Accuracy after sorting by the LATER herald (not a message): {(r[herald] == m[herald]).mean():.4f}")

**Reading the result.** The first number is what a receiver in the past could actually use. It should sit at 0.5. The last number is high only because the runs were sorted using information produced after the record was made.